# Fraud Detection EDA Notebook + DVC Assessment

## Bối cảnh
Notebook này được viết để phục vụ **EDA thực chiến** cho bài toán **fraud transaction detection** trong repository hiện tại.  
Với repo này, assumption chính là bạn đang dùng bộ dữ liệu theo cấu trúc **IEEE-CIS Fraud Detection**:

- `raw_data/train_transaction.csv.gz`
- `raw_data/train_identity.csv`
- target mặc định: `isFraud`
- khóa merge mặc định: `TransactionID`
- cột thời gian tương đối mặc định: `TransactionDT`

## Mục tiêu notebook
1. Làm EDA kỹ và bám sát bài toán fraud detection  
2. Tập trung vào các điểm ảnh hưởng trực tiếp tới modeling:
   - **Recall**
   - **Precision**
   - **F1-score**
   - **PR-AUC**
3. Đánh giá xem project hiện tại có nên dùng **DVC** hay không
4. Nếu có, đưa ra lộ trình triển khai **DVC tối thiểu nhưng thực tế** cho repo sinh viên theo hướng MLOps

## Ghi chú quan trọng
- Notebook ưu tiên **tính chạy được** hơn là hard-code theo một file cụ thể
- Setup cell sẽ tự tìm file dữ liệu trong một số path phổ biến
- Nếu máy yếu RAM, bạn có thể tạm đặt `LOAD_IDENTITY = False` để chạy EDA transaction-only trước
- Notebook **chưa tách riêng** data quality checks / feature engineering thành module riêng, nhưng vẫn chỉ ra các tín hiệu quan trọng cho modeling


In [ ]:
from __future__ import annotations

import gc
import io
import math
import os
import re
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# Display options
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Plot style
sns.set_theme(style="whitegrid", context="notebook")

# Global config
RANDOM_STATE = 42
PLOT_SAMPLE_SIZE = 120_000
CORR_SAMPLE_ROWS = 200_000
MAX_HEATMAP_FEATURES = 25
LOAD_IDENTITY = True  # Nếu máy yếu RAM, đổi thành False để chỉ chạy train_transaction

print("Libraries imported successfully.")


## Setup & Data Discovery

Cell dưới đây làm 4 việc:

1. Tìm repo root và tìm file dữ liệu theo tên phổ biến  
2. Đọc `train_transaction` và (nếu có) `train_identity`  
3. Merge bằng `TransactionID` nếu identity file tồn tại  
4. Tự nhận diện:
   - target column
   - id column
   - time column
   - numerical / categorical features

Ngoài ra, notebook có thêm một số helper để:
- giảm memory usage
- sample hợp lý cho plotting
- gom feature theo nhóm prefix (Transaction / card / addr / C / D / M / V / id / Device ...)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "raw_data").exists():
            return candidate
    return start

def get_git_root(start: Path) -> Path:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=start,
            capture_output=True,
            text=True,
            check=True,
        )
        return Path(result.stdout.strip())
    except Exception:
        return start

def find_first_existing_path(root: Path, candidates: list[str | Path]) -> Path | None:
    for rel in candidates:
        path = root / Path(rel)
        if path.exists():
            return path

    candidate_names = list(dict.fromkeys([Path(c).name for c in candidates]))
    for name in candidate_names:
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None

def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.replace("-", "_") for c in out.columns]
    return out

def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)

def reduce_mem_usage(df: pd.DataFrame, category_threshold: float = 0.02) -> pd.DataFrame:
    start_mem = memory_usage_mb(df)

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif pd.api.types.is_bool_dtype(col_type):
            df[col] = df[col].astype("int8")
        elif pd.api.types.is_object_dtype(col_type):
            nunique = df[col].nunique(dropna=False)
            if 0 < nunique < max(50, int(len(df) * category_threshold)):
                df[col] = df[col].astype("category")

    end_mem = memory_usage_mb(df)
    reduction = ((start_mem - end_mem) / max(start_mem, 1e-9)) * 100
    print(f"Memory usage: {start_mem:,.2f} MB -> {end_mem:,.2f} MB ({reduction:,.1f}% reduction)")
    return df

def infer_feature_family(col: str) -> str:
    if col in {"TransactionID", "TransactionDT", "TransactionAmt"}:
        return "Transaction"
    if col.startswith("card"):
        return "card"
    if col.startswith("addr"):
        return "addr"
    if col.startswith("dist"):
        return "dist"
    if col.startswith("P_email"):
        return "P_emaildomain"
    if col.startswith("R_email"):
        return "R_emaildomain"
    if re.fullmatch(r"C\d+", col):
        return "C"
    if re.fullmatch(r"D\d+", col):
        return "D"
    if re.fullmatch(r"M\d+", col):
        return "M"
    if re.fullmatch(r"V\d+", col):
        return "V"
    if re.fullmatch(r"id_?\d+", col):
        return "id"
    if col.startswith("Device"):
        return "Device"
    if col == "ProductCD":
        return "Product"
    if col == "isFraud":
        return "Target"
    return col.split("_")[0]

def detect_target_column(df: pd.DataFrame) -> str | None:
    candidates = [
        "isFraud",
        "fraud",
        "is_fraud",
        "target",
        "label",
        "Class",
        "class",
        "y",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def detect_time_column(df: pd.DataFrame) -> str | None:
    preferred = [
        "TransactionDT",
        "transaction_time",
        "timestamp",
        "event_time",
        "time",
        "datetime",
        "date",
    ]
    for c in preferred:
        if c in df.columns:
            return c
    for c in df.columns:
        cl = c.lower()
        if "time" in cl or "date" in cl:
            return c
    return None

def fraud_focused_sample(
    df: pd.DataFrame,
    target_col: str,
    max_rows: int = 120_000,
    positive_cap_ratio: float = 0.25,
    random_state: int = 42,
) -> pd.DataFrame:
    if target_col not in df.columns or len(df) <= max_rows:
        return df.copy()

    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]

    if pos.empty or neg.empty:
        return df.sample(n=min(len(df), max_rows), random_state=random_state).copy()

    pos_cap = int(max_rows * positive_cap_ratio)
    if len(pos) <= pos_cap:
        pos_sample = pos
        neg_n = max_rows - len(pos_sample)
    else:
        pos_sample = pos.sample(n=pos_cap, random_state=random_state)
        neg_n = max_rows - pos_cap

    neg_sample = neg.sample(n=min(len(neg), neg_n), random_state=random_state)
    out = pd.concat([pos_sample, neg_sample], axis=0).sample(frac=1.0, random_state=random_state)
    return out

def numeric_target_correlations(
    df: pd.DataFrame,
    numeric_cols: list[str],
    target_col: str,
    sample_rows: int = 200_000,
    random_state: int = 42,
) -> pd.Series:
    cols = [c for c in numeric_cols if c != target_col and df[c].notna().sum() > 0]
    if not cols:
        return pd.Series(dtype=float)

    use_cols = cols + [target_col]
    if len(df) > sample_rows:
        base = df[use_cols].sample(n=sample_rows, random_state=random_state)
    else:
        base = df[use_cols].copy()

    corr = base[cols].corrwith(base[target_col])
    corr = corr.dropna()
    return corr.sort_values(key=lambda s: s.abs(), ascending=False)

def choose_numeric_features(
    df: pd.DataFrame,
    numeric_feature_cols: list[str],
    target_corr: pd.Series,
    time_col: str | None,
    top_n: int = 10,
) -> list[str]:
    priority = [
        "TransactionAmt",
        time_col,
        "card1",
        "card2",
        "card3",
        "card5",
        "addr1",
        "addr2",
        "dist1",
        "dist2",
    ]
    priority = [c for c in priority if c and c in numeric_feature_cols]
    ranked = [c for c in target_corr.index.tolist() if c in numeric_feature_cols]
    chosen = list(dict.fromkeys(priority + ranked))
    return chosen[:top_n]

def choose_categorical_features(
    df: pd.DataFrame,
    categorical_cols: list[str],
    target_col: str,
    max_features: int = 6,
) -> list[str]:
    priority = [
        "ProductCD",
        "card4",
        "card6",
        "P_emaildomain",
        "R_emaildomain",
        "DeviceType",
        "M1",
        "M2",
        "M3",
        "M4",
        "M5",
        "M6",
        "M7",
        "M8",
        "M9",
    ]
    available_priority = [c for c in priority if c in categorical_cols]

    low_card = [
        c for c in categorical_cols
        if c not in available_priority and df[c].nunique(dropna=False) <= 20 and c != target_col
    ]
    return (available_priority + low_card)[:max_features]

def summarize_non_missing_identity(df: pd.DataFrame) -> float | None:
    identity_cols = [c for c in df.columns if infer_feature_family(c) in {"id", "Device"}]
    if not identity_cols:
        return None
    return df[identity_cols].notna().any(axis=1).mean()


In [ ]:
REPO_ROOT = find_repo_root()
GIT_ROOT = get_git_root(REPO_ROOT)

transaction_candidates = [
    "raw_data/train_transaction.csv.gz",
    "raw_data/train_transaction.csv",
    "data/raw/train_transaction.csv.gz",
    "data/raw/train_transaction.csv",
    "train_transaction.csv.gz",
    "train_transaction.csv",
]

identity_candidates = [
    "raw_data/train_identity.csv",
    "data/raw/train_identity.csv",
    "train_identity.csv",
]

TRANSACTION_PATH = find_first_existing_path(GIT_ROOT, transaction_candidates)
IDENTITY_PATH = find_first_existing_path(GIT_ROOT, identity_candidates)

if TRANSACTION_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy file train_transaction. Hãy kiểm tra lại path dữ liệu hoặc sửa candidate paths ở cell setup."
    )

print(f"Repo root detected     : {GIT_ROOT}")
print(f"Transaction file       : {TRANSACTION_PATH}")
print(f"Identity file          : {IDENTITY_PATH if IDENTITY_PATH else 'Not found'}")
print(f"LOAD_IDENTITY          : {LOAD_IDENTITY}")

tx_df = pd.read_csv(TRANSACTION_PATH, low_memory=False)
tx_df = normalize_column_names(tx_df)
tx_df = reduce_mem_usage(tx_df, category_threshold=0.01)

if LOAD_IDENTITY and IDENTITY_PATH is not None:
    id_df = pd.read_csv(IDENTITY_PATH, low_memory=False)
    id_df = normalize_column_names(id_df)
    id_df = reduce_mem_usage(id_df, category_threshold=0.01)

    if "TransactionID" in tx_df.columns and "TransactionID" in id_df.columns:
        df = tx_df.merge(id_df, on="TransactionID", how="left")
        merge_note = "Merged train_transaction + train_identity using TransactionID"
    else:
        df = tx_df.copy()
        merge_note = "Identity file exists but TransactionID not found in both tables => fallback to transaction-only"
    del id_df
    gc.collect()
else:
    df = tx_df.copy()
    merge_note = "Using train_transaction only"

del tx_df
gc.collect()

df = reduce_mem_usage(df, category_threshold=0.01)

target_col = detect_target_column(df)
if target_col is None:
    raise ValueError("Không tìm thấy target column. Hãy sửa detect_target_column() theo dataset thực tế.")

id_col = "TransactionID" if "TransactionID" in df.columns else None
time_col = detect_time_column(df)

numeric_cols = df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
numeric_feature_cols = [c for c in numeric_cols if c not in {target_col, id_col}]
categorical_cols = [c for c in df.columns if c not in numeric_cols and c != id_col]

identity_coverage = summarize_non_missing_identity(df)

print("\n===== Dataset load summary =====")
print(merge_note)
print(f"Shape                   : {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Target column           : {target_col}")
print(f"ID column               : {id_col}")
print(f"Time column             : {time_col}")
print(f"Numeric columns         : {len(numeric_cols):,}")
print(f"Numeric feature columns : {len(numeric_feature_cols):,}")
print(f"Categorical columns     : {len(categorical_cols):,}")
print(f"Approx memory usage     : {memory_usage_mb(df):,.2f} MB")

if identity_coverage is not None:
    print(f"Rows with at least one identity/device field present: {identity_coverage:.2%}")
else:
    print("No identity/device feature family detected after loading.")


## 1) Dataset Overview

Phần này trả lời các câu hỏi nền tảng:

- Dataset có bao nhiêu dòng, bao nhiêu cột?
- Có đúng target không?
- Kiểu dữ liệu ra sao?
- Có đang làm việc trên transaction-only hay merged transaction + identity?
- Các feature đang phân bố theo nhóm prefix như thế nào?

Với IEEE-CIS, việc nhìn theo **feature family** rất quan trọng vì dataset có nhiều masked features như:
- `C*`
- `D*`
- `M*`
- `V*`
- `id_*`
- nhóm `card*`, `addr*`, `dist*`


In [ ]:
overview_table = pd.DataFrame(
    {
        "metric": [
            "n_rows",
            "n_columns",
            "target_column",
            "id_column",
            "time_column",
            "numeric_columns",
            "categorical_columns",
            "merged_identity",
            "memory_usage_mb",
        ],
        "value": [
            df.shape[0],
            df.shape[1],
            target_col,
            id_col,
            time_col,
            len(numeric_cols),
            len(categorical_cols),
            "yes" if "Merged" in merge_note else "no",
            round(memory_usage_mb(df), 2),
        ],
    }
)
display(overview_table)

display(Markdown("### First 5 rows"))
display(df.head())

dtype_summary = (
    df.dtypes.astype(str)
      .value_counts()
      .rename_axis("dtype")
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
display(Markdown("### Dtype summary"))
display(dtype_summary)

feature_family_series = pd.Series([infer_feature_family(c) for c in df.columns], index=df.columns, name="family")
family_summary = (
    feature_family_series.value_counts()
    .rename_axis("family")
    .reset_index(name="n_columns")
    .sort_values(["n_columns", "family"], ascending=[False, True])
)
display(Markdown("### Feature family summary"))
display(family_summary)

missing_pct_all = df.isna().mean() * 100
nunique_all = df.nunique(dropna=False)

family_missing_summary = (
    pd.DataFrame(
        {
            "family": feature_family_series.values,
            "missing_pct": missing_pct_all.values,
        }
    )
    .groupby("family", as_index=False)
    .agg(
        n_columns=("family", "size"),
        avg_missing_pct=("missing_pct", "mean"),
        max_missing_pct=("missing_pct", "max"),
    )
    .sort_values(["avg_missing_pct", "n_columns"], ascending=[False, False])
)
display(Markdown("### Missingness by feature family"))
display(family_missing_summary)

column_profile = (
    pd.DataFrame(
        {
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values,
            "family": feature_family_series.values,
            "missing_pct": missing_pct_all.values,
            "nunique": nunique_all.values,
        }
    )
    .sort_values(["family", "missing_pct", "column"], ascending=[True, False, True])
)
display(Markdown("### Column profile (sample view)"))
display(column_profile.head(50))

buffer = io.StringIO()
df.info(buf=buffer, show_counts=True)
print(buffer.getvalue())


## 2) Target Analysis

Đây là phần cực quan trọng cho fraud detection.

Trong bài toán fraud:
- **class imbalance** gần như luôn là vấn đề trung tâm
- Accuracy thường gây hiểu lầm
- **Recall, Precision, F1-score, PR-AUC** quan trọng hơn nhiều

Đặc biệt:
- nếu fraud rate rất thấp, một model đoán toàn non-fraud vẫn có thể đạt accuracy cao
- baseline **PR-AUC no-skill** xấp xỉ **tỷ lệ positive class**


In [ ]:
target_counts = df[target_col].value_counts(dropna=False).sort_index()
target_ratios = df[target_col].value_counts(normalize=True, dropna=False).sort_index()
target_summary = pd.DataFrame({"count": target_counts, "ratio": target_ratios})
target_summary["ratio_pct"] = target_summary["ratio"] * 100

display(target_summary)

if 1 in target_summary.index:
    positive_ratio = float(target_summary.loc[1, "ratio"])
else:
    # fallback nếu positive class không được encode là 1
    positive_ratio = float(target_summary["ratio"].min())

no_skill_pr_auc = positive_ratio

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

target_plot = target_summary.reset_index()
target_plot = target_plot.rename(columns={target_plot.columns[0]: "class_value"})
sns.barplot(data=target_plot, x="class_value", y="count", ax=axes[0])
axes[0].set_title("Class distribution (count)")
axes[0].set_xlabel(target_col)
axes[0].set_yscale("log")

sns.barplot(data=target_plot, x="class_value", y="ratio_pct", ax=axes[1])
axes[1].set_title("Class distribution (percentage)")
axes[1].set_xlabel(target_col)
axes[1].set_ylabel("Percentage (%)")

for ax in axes:
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f"{height:,.2f}" if ax is axes[1] else f"{int(height):,}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 5),
            textcoords="offset points",
        )

plt.tight_layout()
plt.show()

print(f"Fraud prevalence (positive class rate): {positive_ratio:.4%}")
print(f"No-skill PR-AUC baseline ≈ positive class prevalence: {no_skill_pr_auc:.6f}")
print("Interpretation:")
print("- Accuracy không phải metric chính cho dataset mất cân bằng.")
print("- Recall quan trọng vì bỏ sót fraud thường có cost cao.")
print("- Precision quan trọng để tránh quá nhiều false alarms.")
print("- F1-score cân bằng Recall và Precision.")
print("- PR-AUC đặc biệt hữu ích khi positive class hiếm.")


## 3) Basic Data Inspection (ở mức EDA)

Phần này chưa phải Data Quality module riêng, nhưng đủ sâu để trả lời các câu hỏi thực chiến:

- Missing values nằm ở đâu?
- Duplicate rows / duplicate IDs có tồn tại không?
- Có cột gần như vô dụng không? (constant / gần như all-missing)
- Missingness có khác biệt giữa fraud và non-fraud không?
- Với numerical features, range / quantile / scale có gì đáng chú ý?
- Nếu có categorical features, cardinality và missingness ra sao?


In [ ]:
missing_table = (
    pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_pct": missing_pct_all.values,
            "dtype": df.dtypes.astype(str).values,
            "family": feature_family_series.values,
        }
    )
    .sort_values(["missing_pct", "missing_count"], ascending=[False, False])
)
display(Markdown("### Top columns by missingness"))
display(missing_table.head(30))

print(f"Columns with > 50% missing: {(missing_pct_all > 50).sum():,}")
print(f"Columns with > 80% missing: {(missing_pct_all > 80).sum():,}")
print(f"Columns with > 90% missing: {(missing_pct_all > 90).sum():,}")
print(f"Columns with 0% missing   : {(missing_pct_all == 0).sum():,}")

plt.figure(figsize=(10, 4))
sns.histplot(missing_pct_all, bins=30)
plt.title("Distribution of missing percentage across columns")
plt.xlabel("Missing percentage")
plt.show()

duplicate_rows = int(df.duplicated().sum())
print(f"Duplicate full rows: {duplicate_rows:,}")

if id_col is not None:
    duplicate_ids = int(df[id_col].duplicated().sum())
    print(f"Duplicate {id_col}: {duplicate_ids:,}")
else:
    duplicate_ids = None
    print("No ID column detected, so duplicate ID check is skipped.")

constant_cols = nunique_all[nunique_all <= 1].index.tolist()
print(f"Constant columns (nunique <= 1): {len(constant_cols):,}")
if constant_cols:
    print("Sample constant columns:", constant_cols[:20])

# Quasi-constant check only on low-cardinality categorical columns for speed
quasi_constant_records = []
for col in categorical_cols:
    nunique = int(nunique_all[col])
    if nunique <= 20:
        top_share = float(df[col].value_counts(dropna=False, normalize=True).iloc[0])
        if top_share >= 0.98:
            quasi_constant_records.append(
                {
                    "column": col,
                    "top_value_share": top_share,
                    "nunique": nunique,
                    "missing_pct": float(missing_pct_all[col]),
                }
            )

quasi_constant_table = pd.DataFrame(quasi_constant_records)
if not quasi_constant_table.empty:
    quasi_constant_table = quasi_constant_table.sort_values("top_value_share", ascending=False)
display(Markdown("### Quasi-constant categorical columns (top category >= 98%)"))
display(quasi_constant_table.head(20) if not quasi_constant_table.empty else pd.DataFrame({"info": ["No quasi-constant low-cardinality categorical columns detected at the 98% threshold."]}))

# Missingness gap between fraud and non-fraud
missing_gap_records = []
for col in df.columns:
    fraud_missing = float(df.loc[df[target_col] == 1, col].isna().mean() * 100)
    nonfraud_missing = float(df.loc[df[target_col] == 0, col].isna().mean() * 100)
    missing_gap_records.append(
        {
            "column": col,
            "fraud_missing_pct": fraud_missing,
            "nonfraud_missing_pct": nonfraud_missing,
            "gap_pp": fraud_missing - nonfraud_missing,
            "abs_gap_pp": abs(fraud_missing - nonfraud_missing),
            "family": infer_feature_family(col),
        }
    )

missing_gap_table = pd.DataFrame(missing_gap_records).sort_values("abs_gap_pp", ascending=False)
display(Markdown("### Features whose missingness differs most between fraud and non-fraud"))
display(missing_gap_table.head(20))

numeric_summary = (
    pd.DataFrame(
        {
            "dtype": df[numeric_feature_cols].dtypes.astype(str),
            "missing_pct": df[numeric_feature_cols].isna().mean() * 100,
            "nunique": df[numeric_feature_cols].nunique(dropna=False),
            "mean": df[numeric_feature_cols].mean(numeric_only=True),
            "std": df[numeric_feature_cols].std(numeric_only=True),
            "min": df[numeric_feature_cols].min(numeric_only=True),
            "median": df[numeric_feature_cols].median(numeric_only=True),
            "p95": df[numeric_feature_cols].quantile(0.95, numeric_only=True),
            "p99": df[numeric_feature_cols].quantile(0.99, numeric_only=True),
            "max": df[numeric_feature_cols].max(numeric_only=True),
        }
    )
    .sort_values(["missing_pct", "nunique"], ascending=[False, False])
)
display(Markdown("### Numerical feature summary (sample view)"))
display(numeric_summary.head(30))

if categorical_cols:
    categorical_summary = (
        pd.DataFrame(
            {
                "dtype": df[categorical_cols].dtypes.astype(str),
                "missing_pct": df[categorical_cols].isna().mean() * 100,
                "nunique": df[categorical_cols].nunique(dropna=False),
            }
        )
        .sort_values(["nunique", "missing_pct"], ascending=[True, False])
    )
    display(Markdown("### Categorical feature summary"))
    display(categorical_summary.head(30))

if "TransactionAmt" in df.columns:
    zero_amt = int((df["TransactionAmt"] == 0).sum())
    negative_amt = int((df["TransactionAmt"] < 0).sum())
    print(f"TransactionAmt == 0: {zero_amt:,}")
    print(f"TransactionAmt < 0 : {negative_amt:,}")


## 4) Univariate Analysis

Vì số lượng feature có thể rất lớn, notebook **không vẽ tất cả** để tránh quá nặng.  
Thay vào đó, nó chọn feature theo 2 cách:

1. **Domain-priority features** (ví dụ `TransactionAmt`, `TransactionDT`, `card*`, `addr*`, `dist*`)  
2. **Top numerical features có tương quan mạnh nhất với target** (sample-based)

Mục tiêu:
- nhìn phân phối
- phát hiện skewness mạnh
- phát hiện long tail / heavy tail
- thấy nhanh feature nào có scale bất thường trước khi modeling


In [ ]:
numeric_target_corr = numeric_target_correlations(
    df=df,
    numeric_cols=numeric_feature_cols,
    target_col=target_col,
    sample_rows=CORR_SAMPLE_ROWS,
    random_state=RANDOM_STATE,
)

univariate_features = choose_numeric_features(
    df=df,
    numeric_feature_cols=numeric_feature_cols,
    target_corr=numeric_target_corr,
    time_col=time_col,
    top_n=10,
)

display(Markdown("### Selected numerical features for univariate analysis"))
display(pd.DataFrame({"feature": univariate_features}))

if not univariate_features:
    print("No numerical features were selected for univariate analysis.")
else:
    univariate_stats = pd.DataFrame(
        {
            "missing_pct": df[univariate_features].isna().mean() * 100,
            "nunique": df[univariate_features].nunique(dropna=False),
            "skewness": df[univariate_features].skew(numeric_only=True),
            "mean": df[univariate_features].mean(numeric_only=True),
            "median": df[univariate_features].median(numeric_only=True),
            "p95": df[univariate_features].quantile(0.95, numeric_only=True),
            "p99": df[univariate_features].quantile(0.99, numeric_only=True),
        }
    ).sort_values("skewness", ascending=False)

    display(Markdown("### Univariate statistics"))
    display(univariate_stats)

    plot_univariate_df = (
        df[univariate_features].sample(n=min(len(df), PLOT_SAMPLE_SIZE), random_state=RANDOM_STATE)
        if len(df) > PLOT_SAMPLE_SIZE
        else df[univariate_features].copy()
    )

    fig, axes = plt.subplots(len(univariate_features), 2, figsize=(16, max(4, 4 * len(univariate_features))))
    axes = np.array(axes).reshape(len(univariate_features), 2)

    for i, col in enumerate(univariate_features):
        series = plot_univariate_df[col].dropna()

        if series.empty:
            axes[i, 0].set_visible(False)
            axes[i, 1].set_visible(False)
            continue

        plot_series = series.copy()
        display_label = col

        if series.min() >= 0 and series.skew() > 2:
            plot_series = np.log1p(series)
            display_label = f"log1p({col})"

        sns.histplot(plot_series, bins=60, kde=True, ax=axes[i, 0])
        axes[i, 0].set_title(f"Histogram / KDE - {display_label}")

        sns.boxplot(x=plot_series, ax=axes[i, 1], orient="h")
        axes[i, 1].set_title(f"Boxplot - {display_label}")

    plt.tight_layout()
    plt.show()

print("Practical reading notes:")
print("- Feature rất lệch phải (right-skewed) thường cần log transform / robust scaling / tree model handling.")
print("- Với fraud detection, extreme values không phải lúc nào cũng là noise; nhiều outlier là tín hiệu có ích.")
print("- Các masked features có phân phối khác thường vẫn đáng giữ lại cho modeling, miễn là đánh giá bằng validation đúng.")


## 5) Fraud vs Non-Fraud Comparison

Phần này đi thẳng vào câu hỏi modeling:

- Feature nào có khả năng tách fraud khỏi non-fraud?
- Có feature nào khác biệt rõ ở median / spread / tail?
- Với categorical features, category nào có fraud rate cao bất thường?

Để biểu đồ nhẹ hơn nhưng vẫn giữ insight:
- phần plotting dùng **fraud-focused sample**
- phần summary table vẫn cố gắng tính trên full dataset


In [ ]:
compare_numeric_features = list(
    dict.fromkeys(
        [c for c in ["TransactionAmt", time_col] if c in univariate_features] + numeric_target_corr.head(6).index.tolist()
    )
)[:6]

numeric_compare_records = []
for col in compare_numeric_features:
    fraud_vals = df.loc[df[target_col] == 1, col].dropna()
    nonfraud_vals = df.loc[df[target_col] == 0, col].dropna()

    numeric_compare_records.append(
        {
            "feature": col,
            "missing_pct": float(df[col].isna().mean() * 100),
            "median_nonfraud": float(nonfraud_vals.median()) if len(nonfraud_vals) else np.nan,
            "median_fraud": float(fraud_vals.median()) if len(fraud_vals) else np.nan,
            "mean_nonfraud": float(nonfraud_vals.mean()) if len(nonfraud_vals) else np.nan,
            "mean_fraud": float(fraud_vals.mean()) if len(fraud_vals) else np.nan,
            "std_nonfraud": float(nonfraud_vals.std()) if len(nonfraud_vals) else np.nan,
            "std_fraud": float(fraud_vals.std()) if len(fraud_vals) else np.nan,
            "abs_target_corr": float(abs(numeric_target_corr.get(col, np.nan))),
        }
    )

numeric_compare_table = pd.DataFrame(numeric_compare_records)
if not numeric_compare_table.empty:
    numeric_compare_table = numeric_compare_table.sort_values("abs_target_corr", ascending=False)

display(Markdown("### Numeric feature comparison by class"))
display(numeric_compare_table if not numeric_compare_table.empty else pd.DataFrame({"info": ["No numeric features were selected for class comparison."]}))

if compare_numeric_features:
    plot_compare_numeric_df = fraud_focused_sample(
        df[[target_col] + compare_numeric_features].copy(),
        target_col=target_col,
        max_rows=PLOT_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )

    fig, axes = plt.subplots(len(compare_numeric_features), 2, figsize=(16, max(4, 4 * len(compare_numeric_features))))
    axes = np.array(axes).reshape(len(compare_numeric_features), 2)

    for i, col in enumerate(compare_numeric_features):
        work = plot_compare_numeric_df[[target_col, col]].dropna()
        if work.empty:
            axes[i, 0].set_visible(False)
            axes[i, 1].set_visible(False)
            continue

        display_col = col
        work_plot = work.copy()

        if work[col].min() >= 0 and work[col].skew() > 2:
            work_plot[col] = np.log1p(work[col])
            display_col = f"log1p({col})"

        sns.histplot(
            data=work_plot,
            x=col,
            hue=target_col,
            bins=60,
            stat="density",
            common_norm=False,
            element="step",
            ax=axes[i, 0],
        )
        axes[i, 0].set_title(f"Fraud vs Non-Fraud density - {display_col}")

        sns.boxplot(
            data=work_plot,
            x=target_col,
            y=col,
            showfliers=False,
            ax=axes[i, 1],
        )
        axes[i, 1].set_title(f"Fraud vs Non-Fraud boxplot - {display_col}")

    plt.tight_layout()
    plt.show()


In [ ]:
categorical_features_to_analyze = choose_categorical_features(
    df=df,
    categorical_cols=categorical_cols,
    target_col=target_col,
    max_features=6,
)

display(Markdown("### Selected categorical features"))
display(pd.DataFrame({"feature": categorical_features_to_analyze}))

for col in categorical_features_to_analyze:
    tmp = df[[col, target_col]].copy()
    tmp[col] = tmp[col].astype("string").fillna("Missing")

    top_categories = tmp[col].value_counts(dropna=False).head(10).index
    plot_table = (
        tmp[tmp[col].isin(top_categories)]
        .groupby(col)[target_col]
        .agg(count="size", fraud_rate="mean")
        .reset_index()
        .sort_values("count", ascending=False)
    )
    plot_table["fraud_rate_pct"] = plot_table["fraud_rate"] * 100
    plot_table["lift_vs_base_rate"] = plot_table["fraud_rate"] / max(positive_ratio, 1e-9)

    display(Markdown(f"#### {col} - top categories by count with fraud rate"))
    display(plot_table)

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    sns.barplot(data=plot_table, x=col, y="count", ax=axes[0])
    axes[0].set_title(f"{col} - transaction count")
    axes[0].tick_params(axis="x", rotation=45)

    sns.barplot(data=plot_table, x=col, y="fraud_rate_pct", ax=axes[1])
    axes[1].set_title(f"{col} - fraud rate (%)")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

print("Interpretation notes:")
print("- Hãy nhìn đồng thời cả count và fraud rate; rate cao nhưng count quá nhỏ có thể chỉ là noise.")
print("- Những category có fraud rate cao ổn định là ứng viên tốt cho feature engineering / encoding sau này.")
print("- Với dataset loại này, missing category hoặc category hiếm đôi khi lại là tín hiệu fraud mạnh.")


## 6) Correlation Analysis

Với fraud dataset nhiều feature, heatmap full matrix thường quá rối.  
Notebook này làm 2 việc thực dụng hơn:

1. Liệt kê **top numerical features có correlation với target**  
2. Vẽ heatmap cho **một tập feature đã chọn lọc**  
3. Liệt kê **các cặp feature tương quan mạnh với nhau** để cảnh báo redundancy / multicollinearity

Lưu ý:
- correlation không đồng nghĩa causal
- với tree model, multicollinearity ít nghiêm trọng hơn linear model
- nhưng correlation cao vẫn hữu ích để hiểu cấu trúc feature groups


In [ ]:
display(Markdown("### Top numerical features correlated with target"))
target_corr_table = (
    pd.DataFrame(
        {
            "feature": numeric_target_corr.index,
            "corr_with_target": numeric_target_corr.values,
            "abs_corr_with_target": np.abs(numeric_target_corr.values),
            "missing_pct": [float(missing_pct_all.get(c, np.nan)) for c in numeric_target_corr.index],
            "family": [infer_feature_family(c) for c in numeric_target_corr.index],
        }
    )
    .sort_values("abs_corr_with_target", ascending=False)
)
display(target_corr_table.head(20) if not target_corr_table.empty else pd.DataFrame({"info": ["No numeric target correlations available."]}))

heatmap_features = list(
    dict.fromkeys(
        [c for c in ["TransactionAmt", time_col] if c in numeric_feature_cols]
        + target_corr_table["feature"].head(MAX_HEATMAP_FEATURES).tolist()
    )
)[:MAX_HEATMAP_FEATURES]

if len(heatmap_features) >= 2:
    corr_base = (
        df[heatmap_features].sample(n=min(len(df), CORR_SAMPLE_ROWS), random_state=RANDOM_STATE)
        if len(df) > CORR_SAMPLE_ROWS
        else df[heatmap_features].copy()
    )

    corr_matrix = corr_base.corr(numeric_only=True)

    plt.figure(figsize=(14, 12))
    sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
    plt.title("Correlation heatmap for selected numerical features")
    plt.show()

    upper_mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    pairwise_corr = (
        corr_matrix.abs()
        .where(upper_mask)
        .stack()
        .reset_index()
    )
    pairwise_corr.columns = ["feature_1", "feature_2", "abs_corr"]
    pairwise_corr = pairwise_corr.sort_values("abs_corr", ascending=False)

    display(Markdown("### Strongest correlated feature pairs"))
    display(pairwise_corr.head(20))

    strong_pairs = pairwise_corr[pairwise_corr["abs_corr"] >= 0.80]
    print(f"Number of selected feature pairs with |corr| >= 0.80: {len(strong_pairs):,}")
else:
    corr_matrix = pd.DataFrame()
    pairwise_corr = pd.DataFrame(columns=["feature_1", "feature_2", "abs_corr"])
    print("Not enough numeric features for correlation heatmap.")


## 7) Outlier Analysis

Trong fraud detection, outlier là phần cần nhìn cực cẩn thận:

- Outlier **không tự động** là lỗi dữ liệu
- Outlier có thể là **hành vi bất thường** rất hữu ích cho fraud model
- Nếu cắt / winsorize / clip quá sớm, bạn có thể làm giảm **Recall**

Vì vậy ở đây notebook:
1. Dùng IQR để **đo mức outlier**
2. So sánh **outlier rate giữa fraud và non-fraud**
3. Nhấn mạnh insight, chưa vội loại bỏ


In [ ]:
outlier_features = list(
    dict.fromkeys(
        [c for c in ["TransactionAmt", time_col] if c in numeric_feature_cols]
        + target_corr_table["feature"].head(8).tolist()
    )
)[:8]

outlier_records = []
for col in outlier_features:
    s = df[col].dropna()
    if s.empty or s.nunique() < 5:
        continue

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        continue

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)

    overall_rate = float(mask.mean())
    fraud_mask = mask[df[target_col] == 1]
    nonfraud_mask = mask[df[target_col] == 0]

    fraud_outlier_rate = float(fraud_mask.mean()) if len(fraud_mask) else np.nan
    nonfraud_outlier_rate = float(nonfraud_mask.mean()) if len(nonfraud_mask) else np.nan
    fraud_outlier_lift = (
        fraud_outlier_rate / nonfraud_outlier_rate
        if pd.notna(nonfraud_outlier_rate) and nonfraud_outlier_rate > 0
        else np.nan
    )

    outlier_records.append(
        {
            "feature": col,
            "lower_bound": float(lower),
            "upper_bound": float(upper),
            "overall_outlier_rate_pct": overall_rate * 100,
            "fraud_outlier_rate_pct": fraud_outlier_rate * 100 if pd.notna(fraud_outlier_rate) else np.nan,
            "nonfraud_outlier_rate_pct": nonfraud_outlier_rate * 100 if pd.notna(nonfraud_outlier_rate) else np.nan,
            "fraud_outlier_lift": fraud_outlier_lift,
        }
    )

outlier_table = pd.DataFrame(outlier_records)
if not outlier_table.empty:
    outlier_table = outlier_table.sort_values(
        ["fraud_outlier_lift", "overall_outlier_rate_pct"], ascending=[False, False]
    )
display(outlier_table if not outlier_table.empty else pd.DataFrame({"info": ["No outlier table could be computed for the selected features."]}))

top_outlier_plot_features = outlier_table["feature"].head(4).tolist() if not outlier_table.empty else []

if top_outlier_plot_features:
    plot_outlier_df = fraud_focused_sample(
        df[[target_col] + top_outlier_plot_features].copy(),
        target_col=target_col,
        max_rows=PLOT_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )

    fig, axes = plt.subplots(len(top_outlier_plot_features), 1, figsize=(14, max(4, 4 * len(top_outlier_plot_features))))
    axes = np.array(axes).reshape(len(top_outlier_plot_features))

    for i, col in enumerate(top_outlier_plot_features):
        work = plot_outlier_df[[target_col, col]].dropna().copy()

        if work.empty:
            axes[i].set_visible(False)
            continue

        if work[col].min() >= 0 and work[col].skew() > 2:
            work[col] = np.log1p(work[col])
            title_col = f"log1p({col})"
        else:
            title_col = col

        sns.boxplot(data=work, x=target_col, y=col, ax=axes[i], showfliers=True)
        axes[i].set_title(f"Outlier-sensitive view by class - {title_col}")

    plt.tight_layout()
    plt.show()

print("Outlier takeaway:")
print("- Nếu fraud_outlier_lift > 1, outlier xuất hiện ở fraud nhiều hơn non-fraud.")
print("- Đó là dấu hiệu nên thận trọng trước khi clip hoặc remove outliers.")
print("- Với tree-based models, nhiều khi cứ giữ nguyên sẽ tốt hơn cho Recall/PR-AUC.")


## 8) Time-based Analysis

Nếu dataset có cột thời gian, đây là phần không nên bỏ qua trong fraud detection:

- volume giao dịch có drift theo thời gian không?
- fraud count có dồn vào những giai đoạn nhất định không?
- fraud rate có thay đổi theo thời gian không?
- có dấu hiệu cho thấy cần **time-based split** thay vì random split không?

Với IEEE-CIS, `TransactionDT` thường được dùng như **relative time index** (không nhất thiết là calendar timestamp thực).


In [ ]:
if time_col is None:
    print("No time-like column detected => time-based analysis is skipped.")
else:
    time_df = df[[time_col, target_col]].dropna().copy()

    if pd.api.types.is_numeric_dtype(time_df[time_col]):
        time_df["day_since_start"] = (time_df[time_col] // 86400).astype(int)
        time_df["hour_since_start"] = (time_df[time_col] // 3600).astype(int)
        time_df["hour_of_day_proxy"] = (time_df["hour_since_start"] % 24).astype(int)
    else:
        parsed = pd.to_datetime(time_df[time_col], errors="coerce")
        time_df = time_df.loc[parsed.notna()].copy()
        parsed = parsed.loc[parsed.notna()]
        time_df["day_since_start"] = (parsed - parsed.min()).dt.days
        time_df["hour_of_day_proxy"] = parsed.dt.hour

    daily = (
        time_df.groupby("day_since_start")[target_col]
        .agg(transactions="size", frauds="sum", fraud_rate="mean")
        .reset_index()
    )
    daily["fraud_rate_7d_ma"] = daily["fraud_rate"].rolling(7, min_periods=1).mean()

    hourly = (
        time_df.groupby("hour_of_day_proxy")[target_col]
        .agg(transactions="size", fraud_rate="mean")
        .reset_index()
        .sort_values("hour_of_day_proxy")
    )

    decile_drift = time_df.copy()
    decile_drift["time_decile"] = pd.qcut(
        decile_drift[time_col].rank(method="first"),
        q=10,
        labels=False,
        duplicates="drop",
    )
    decile_summary = (
        decile_drift.groupby("time_decile")[target_col]
        .agg(transactions="size", fraud_rate="mean")
        .reset_index()
    )

    display(Markdown("### Fraud rate by time decile"))
    display(decile_summary)

    fig, axes = plt.subplots(3, 1, figsize=(16, 14))

    sns.lineplot(data=daily, x="day_since_start", y="transactions", ax=axes[0])
    axes[0].set_title("Transactions over time")
    axes[0].set_xlabel("Day since start")

    sns.lineplot(data=daily, x="day_since_start", y="frauds", ax=axes[1])
    axes[1].set_title("Fraud count over time")
    axes[1].set_xlabel("Day since start")

    sns.lineplot(data=daily, x="day_since_start", y="fraud_rate_7d_ma", ax=axes[2], label="7-day moving average")
    sns.lineplot(data=hourly, x="hour_of_day_proxy", y="fraud_rate", ax=axes[2], label="Hour-of-day fraud rate")
    axes[2].set_title("Fraud rate over time")
    axes[2].set_xlabel("Day since start / Hour of day proxy")

    plt.tight_layout()
    plt.show()

    print("Time-based interpretation:")
    print("- Nếu fraud rate drift theo thời gian, nên ưu tiên validation theo time split.")
    print("- Nếu fraud dồn theo khung giờ/ngày, time-derived features có thể rất hữu ích.")
    print("- Random split có thể quá lạc quan nếu temporal drift mạnh.")


## 9) EDA Summary

Cell dưới đây sinh ra phần tóm tắt insight quan trọng nhất từ EDA để bạn có thể copy nhanh sang report / README / báo cáo nhóm.


In [ ]:
summary_points = []

summary_points.append(
    f"Dataset used for EDA has {df.shape[0]:,} rows and {df.shape[1]:,} columns "
    + ("after merging transaction + identity tables." if 'Merged' in merge_note else "using transaction table only.")
)

summary_points.append(
    f"Fraud prevalence is {positive_ratio:.4%}, so a no-skill PR-AUC is only about {no_skill_pr_auc:.6f}."
)

summary_points.append(
    f"There are {(missing_pct_all > 50).sum():,} columns with >50% missing values and {(missing_pct_all > 90).sum():,} columns with >90% missing values."
)

if duplicate_rows > 0:
    summary_points.append(f"Found {duplicate_rows:,} duplicate full rows; verify whether these are real repeated transactions or data artifacts.")
else:
    summary_points.append("No duplicate full rows were detected.")

if duplicate_ids is not None:
    if duplicate_ids > 0:
        summary_points.append(f"Found {duplicate_ids:,} duplicate IDs in {id_col}; investigate before modeling.")
    else:
        summary_points.append(f"No duplicate {id_col} values were detected.")

if identity_coverage is not None:
    summary_points.append(
        f"About {identity_coverage:.2%} of transactions have at least one identity/device field present after merge."
    )

top_corr_features = target_corr_table["feature"].head(5).tolist()
if top_corr_features:
    summary_points.append(
        "Numerical features with strongest target association (sample-based) include: "
        + ", ".join(top_corr_features)
        + "."
    )

top_missing_gap = missing_gap_table.head(5)["column"].tolist()
if top_missing_gap:
    summary_points.append(
        "Missingness itself may be predictive because features such as "
        + ", ".join(top_missing_gap)
        + " show large missingness gaps between fraud and non-fraud."
    )

if not pairwise_corr.empty:
    strong_pair_sample = pairwise_corr.head(3).apply(lambda r: f"{r['feature_1']}~{r['feature_2']} (|corr|={r['abs_corr']:.2f})", axis=1).tolist()
    summary_points.append(
        "Strongly correlated feature pairs were found, for example: " + "; ".join(strong_pair_sample) + "."
    )

if not outlier_table.empty:
    suspicious_outlier_features = outlier_table.head(3)["feature"].tolist()
    summary_points.append(
        "Outlier-heavy features that deserve extra attention include: " + ", ".join(suspicious_outlier_features) + "."
    )

if time_col is not None:
    summary_points.append(
        f"Time-like feature `{time_col}` exists, so temporal drift should be checked before deciding train/validation split strategy."
    )

modeling_implications = [
    "Do not optimize accuracy first; prioritize Recall, Precision, F1-score, and especially PR-AUC.",
    "Consider time-aware validation if fraud rate or feature distribution drifts over time.",
    "High-missing columns should not be dropped blindly; missingness can itself encode fraud signal.",
    "Outliers may be signal, not noise; clipping/removal should be validated against Recall and PR-AUC.",
    "Category-level fraud rate analysis suggests some categorical variables may benefit from target-aware encoding or grouped rare-category handling in later modeling.",
]

summary_md = "## Key EDA Findings\n" + "\n".join([f"- {p}" for p in summary_points])
summary_md += "\n\n## Modeling Implications\n" + "\n".join([f"- {p}" for p in modeling_implications])

display(Markdown(summary_md))


# 10) DVC Assessment for This Repo

Phần dưới đây **không trả lời lý thuyết chung chung**, mà cố gắng bám sát repo hiện tại:

- repo đang có file dữ liệu gì?
- kích thước data hiện tại có đáng để đưa sang DVC chưa?
- repo đã ở giai đoạn nào?
- kết luận hợp lý nhất là:
  1. **Nên dùng DVC ngay**
  2. **Nên làm EDA trước, DVC sau**
  3. **Chưa cần DVC ở giai đoạn này**


In [ ]:
def safe_git_commit_count(repo_root: Path) -> int | None:
    try:
        result = subprocess.run(
            ["git", "rev-list", "--count", "HEAD"],
            cwd=repo_root,
            capture_output=True,
            text=True,
            check=True,
        )
        return int(result.stdout.strip())
    except Exception:
        return None

raw_data_dir = GIT_ROOT / "raw_data"
raw_data_files = []

if raw_data_dir.exists():
    for p in sorted(raw_data_dir.rglob("*")):
        if p.is_file():
            raw_data_files.append(
                {
                    "path": str(p.relative_to(GIT_ROOT)),
                    "size_mb": round(p.stat().st_size / (1024 ** 2), 2),
                    "suffix": "".join(p.suffixes) if p.suffixes else p.suffix,
                }
            )

raw_data_table = pd.DataFrame(raw_data_files).sort_values("size_mb", ascending=False) if raw_data_files else pd.DataFrame()
display(Markdown("### Data files currently found in repo"))
display(raw_data_table)

total_raw_data_mb = float(raw_data_table["size_mb"].sum()) if not raw_data_table.empty else 0.0
largest_file_mb = float(raw_data_table["size_mb"].max()) if not raw_data_table.empty else 0.0

dvc_initialized = (GIT_ROOT / ".dvc").exists()
has_dvc_yaml = (GIT_ROOT / "dvc.yaml").exists()

non_notebook_ml_dirs = [
    p for p in [
        "src",
        "data/interim",
        "data/processed",
        "data/splits",
        "models",
        "artifacts",
    ]
    if (GIT_ROOT / p).exists()
]

commit_count = safe_git_commit_count(GIT_ROOT)

tracked_large_table = pd.DataFrame()
try:
    git_files = subprocess.run(
        ["git", "ls-files"],
        cwd=GIT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.splitlines()

    tracked_large = []
    for rel in git_files:
        p = GIT_ROOT / rel
        if p.exists() and p.is_file():
            size_mb = p.stat().st_size / (1024 ** 2)
            if size_mb >= 10:
                tracked_large.append(
                    {
                        "git_tracked_large_file": rel,
                        "size_mb": round(size_mb, 2),
                    }
                )

    if tracked_large:
        tracked_large_table = pd.DataFrame(tracked_large).sort_values("size_mb", ascending=False)
except Exception:
    pass

display(Markdown("### Git-tracked large files (>= 10 MB)"))
display(tracked_large_table if not tracked_large_table.empty else pd.DataFrame({"info": ["No git-tracked large files detected or git metadata unavailable"]}))

signals = pd.DataFrame(
    {
        "signal": [
            "total_raw_data_mb",
            "largest_single_file_mb",
            "dvc_initialized",
            "has_dvc_yaml",
            "non_notebook_ml_dirs_detected",
            "git_commit_count",
        ],
        "value": [
            round(total_raw_data_mb, 2),
            round(largest_file_mb, 2),
            dvc_initialized,
            has_dvc_yaml,
            ", ".join(non_notebook_ml_dirs) if non_notebook_ml_dirs else "None",
            commit_count,
        ],
    }
)
display(Markdown("### Repo signals used for DVC decision"))
display(signals)

if dvc_initialized:
    dvc_decision = "1. Nên dùng DVC ngay (thực tế repo đã có DVC hoặc đang ở giai đoạn đủ chín để dùng DVC)"
elif total_raw_data_mb >= 80 and (commit_count is None or commit_count <= 3) and len(non_notebook_ml_dirs) == 0:
    dvc_decision = "2. Nên làm EDA trước, DVC sau"
elif total_raw_data_mb >= 80:
    dvc_decision = "1. Nên dùng DVC ngay"
else:
    dvc_decision = "3. Chưa cần DVC ở giai đoạn này"

reason_lines = []

if total_raw_data_mb >= 80:
    reason_lines.append(
        f"raw_data hiện đã khoảng {total_raw_data_mb:.2f} MB, tức là đủ lớn để Git-only trở nên bất tiện khi project bắt đầu có thêm processed data / models."
    )
else:
    reason_lines.append(
        f"raw_data hiện chỉ khoảng {total_raw_data_mb:.2f} MB, nên áp lực dùng DVC chưa quá cao."
    )

if largest_file_mb >= 50:
    reason_lines.append(
        f"Có file đơn lẻ khoảng {largest_file_mb:.2f} MB; đây là dấu hiệu không nên tiếp tục đẩy thêm artifact lớn trực tiếp vào Git."
    )

if commit_count is not None and commit_count <= 3:
    reason_lines.append(
        "Repo vẫn ở giai đoạn rất sớm (ít commit), nên còn dễ chốt cấu trúc thư mục trước khi formalize DVC workflow."
    )

if len(non_notebook_ml_dirs) == 0:
    reason_lines.append(
        "Hiện repo gần như chưa có pipeline/data-processing/model-artifact structure rõ ràng, nên chưa cần tạo dvc.yaml ngay."
    )
else:
    reason_lines.append(
        "Repo đã bắt đầu có dấu hiệu pipeline/model artifacts, nên lợi ích của DVC tăng rõ rệt."
    )

decision_md = f"## DVC Decision\n**{dvc_decision}**\n\n" + "\n".join([f"- {x}" for x in reason_lines])
display(Markdown(decision_md))


## Practical DVC Plan cho repo hiện tại

### Kết luận trong notebook này
**Khuyến nghị hiện tại: `2. Nên làm EDA trước, DVC sau`.**

Nói cách khác:
- **không cần chặn EDA chỉ để cài DVC trước**
- nhưng **ngay sau khi notebook này ổn định**, bạn nên bắt đầu đưa data lớn ra khỏi Git tracking thường

### Vì sao không chọn “chưa cần DVC”?
Vì repo này đã có `raw_data/` khá nặng. Nếu đi tiếp theo hướng MLOps mà vẫn để:
- raw dataset
- merged dataset
- processed dataset
- train/validation splits
- model artifacts

đều nằm trong Git thường, repo sẽ phình rất nhanh và khó cộng tác.

### Vì sao chưa cần `dvc.yaml` ngay?
Vì ở bước hiện tại, mục tiêu chính vẫn là:
- ổn định EDA
- chốt cấu trúc thư mục
- xác định pipeline data/modeling sau EDA

Ở giai đoạn này, **DVC tối thiểu** là đủ:
- `dvc init`
- `dvc add`
- cấu hình remote
- `dvc push / pull`

`dvc.yaml` chỉ nên thêm khi preprocessing/train/evaluate scripts đã tương đối ổn.

---

## DVC sẽ giúp gì cho project fraud detection này?

1. **Version raw dataset và processed dataset rõ ràng**
   - rất hữu ích khi bạn thử nhiều cách merge / filtering / split khác nhau

2. **Liên kết code version với data version**
   - khi so sánh model A và model B, bạn biết chắc chúng train trên cùng data snapshot hay không

3. **Tránh đẩy file lớn trực tiếp vào Git**
   - đặc biệt quan trọng khi bắt đầu lưu:
     - `data/processed/*.parquet`
     - `models/*.pkl`
     - feature tables lớn
     - validation artifacts

4. **Hỗ trợ cộng tác nhóm**
   - bạn pull đúng phiên bản data tương ứng với code branch/commit

---

## Những gì nên track bằng DVC

### Nên track bằng DVC
- `raw_data/`
- `data/interim/`
- `data/processed/`
- `data/splits/`
- `models/`
- các artifact lớn (embedding, feature store snapshot, scored outputs lớn)

### Chỉ nên commit bằng Git thường
- notebook `.ipynb`
- code trong `src/`
- `README.md`
- `requirements.txt`
- `.dvc/`
- `*.dvc`
- `.dvcignore`
- `dvc.yaml`
- `params.yaml`
- file config nhỏ (`.yaml`, `.json`, `.toml`)
- báo cáo / hình nhỏ nếu không quá nặng

### Không nên track bằng Git lẫn DVC
- cache tạm
- output sinh ra có thể tái tạo dễ dàng
- `.ipynb_checkpoints/`
- `__pycache__/`
- `.venv/`

---

## Cấu trúc thư mục tối thiểu đề xuất

```text
Fraud-Detection/
├── raw_data/                        # DVC tracked
│   ├── train_transaction.csv.gz
│   ├── train_identity.csv
│   ├── test_transaction.csv.gz
│   ├── test_identity.csv
│   └── sample_submission.csv
├── notebooks/
│   └── 01_eda_fraud_detection.ipynb
├── src/
│   ├── data/
│   │   ├── load_data.py
│   │   └── preprocess.py
│   ├── modeling/
│   │   ├── train.py
│   │   └── evaluate.py
│   └── utils/
├── data/
│   ├── interim/                    # DVC tracked when created
│   ├── processed/                  # DVC tracked when created
│   └── splits/                     # DVC tracked when created
├── models/                         # DVC tracked when created
├── reports/
│   └── figures/
├── .dvc/
├── .dvcignore
├── dvc.yaml                        # thêm sau khi pipeline ổn định
├── params.yaml                     # thêm sau khi pipeline ổn định
├── requirements.txt
└── README.md
```

---

## Cách triển khai DVC tối thiểu nhưng thực tế

### 1) Cài DVC
```bash
pip install dvc
```

> Nếu sau này dùng remote S3/GCS/... thì cài thêm optional dependency tương ứng.

### 2) Init DVC trong repo
```bash
dvc init
git add .dvc/ .dvcignore
git commit -m "Initialize DVC"
```

### 3) Vì `raw_data/` hiện đang nằm trong Git thường, hãy bỏ tracking Git trước
```bash
git rm -r --cached raw_data
```

> Lệnh này bỏ tracking trong Git index nhưng **không xóa file local**.

### 4) Cho DVC quản lý dataset
```bash
dvc add raw_data
git add raw_data.dvc .gitignore
git commit -m "Track raw_data with DVC"
```

### 5) Chọn remote storage phù hợp cho nhóm sinh viên

#### Phương án khuyến nghị (đơn giản nhất)
Dùng **shared folder remote**:
- một thư mục đồng bộ qua OneDrive / Google Drive Desktop / Dropbox
- hoặc một shared folder nội bộ lab/team

Ví dụ:
```bash
dvc remote add -d storage /ABSOLUTE/PATH/TO/shared-fraud-dvc-storage
git add .dvc/config
git commit -m "Configure DVC remote"
```

Ưu điểm:
- setup rất nhanh
- không cần bucket/cloud credentials phức tạp
- phù hợp nhóm sinh viên

#### Nếu nhóm đã có object storage / MinIO / S3-compatible storage
Khi đó dùng remote kiểu bucket sẽ tốt hơn về lâu dài.  
Nhưng cho project sinh viên, **shared folder remote** thường là điểm khởi đầu hợp lý nhất.

### 6) Push dữ liệu lên remote
```bash
dvc push
```

### 7) Khi teammate clone repo
```bash
git clone <your-repo-url>
cd Fraud-Detection
pip install dvc
dvc pull
```

### 8) Khi có processed data / models
```bash
dvc add data/processed
dvc add models
git add data/processed.dvc models.dvc .gitignore
git commit -m "Track processed data and models with DVC"
dvc push
```

---

## File nào vào Git, file nào vào DVC?

### Git
- `.dvc/config`
- `.dvc/`
- `raw_data.dvc`
- `data/processed.dvc`
- `models.dvc`
- notebooks
- source code
- README / config files

### DVC remote storage
- nội dung thực của:
  - `raw_data/`
  - `data/processed/`
  - `models/`

---

## Khi nào nên bắt đầu tích hợp DVC ngay sau notebook này?

Ngay khi bạn làm một trong các việc sau:
- tạo merged dataset lưu ra file
- lưu train/validation split
- train model và lưu `.pkl` / `.joblib`
- có từ 2 phiên bản processed data trở lên
- cần teammate pull đúng data version để reproduce kết quả

Nếu mới chỉ có notebook EDA và chưa tạo artifact mới, có thể **chốt notebook trước**, sau đó tích hợp DVC ngay ở bước tiếp theo.


## Next recommended steps sau notebook này

1. Giữ notebook này là **EDA gốc**  
2. Tạo thêm:
   - `src/data/load_data.py`
   - `src/data/preprocess.py`
   - `src/modeling/train.py`
   - `src/modeling/evaluate.py`
3. Chốt chiến lược validation:
   - ưu tiên kiểm tra **time-aware split**
   - không dùng accuracy làm metric chính
4. Sau khi bắt đầu sinh ra file processed/model:
   - tích hợp DVC tối thiểu như phần trên
